<a href="https://colab.research.google.com/github/CMDDclass/MS697-material/blob/main/Hands-on-session5/Hands-on-session5-BO-assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment


# Bayesian Optimization Assignment: Comparing EI and UCB

## Topic  
The goal of this assignment is to compare two Bayesian Optimization (BO) acquisition strategies — **Expected Improvement (EI)** and **Upper Confidence Bound (UCB)** — across various black-box benchmark functions.

## Objective  
- Visualize how EI and UCB balance **exploration and exploitation** during the optimization process.  
- Measure how quickly each strategy approaches the global optimum on different objective functions.

## Task Overview  
1. Use six standard benchmark functions:  
   - 1D: Forrester, Sinusoidal mixture  
   - 2D: Branin, Goldstein–Price  
   - 3D: Hartmann 3  
   - 6D: Hartmann 6  
2. For each function, run two Bayesian Optimization loops:  
   - One using **Expected Improvement (EI)**  
   - One using **Upper Confidence Bound (UCB)**  
3. Record the **best value found so far** at each iteration.  
4. Plot the progress of both acquisition functions to compare performance.

## Expected Results  
- Each subplot shows the best-so-far curve for **EI (solid line)** and **UCB (dashed line)**.  
- You can visually compare how fast each acquisition strategy converges to the global optimum.  

## Implementation Summary  
- **Model:** Gaussian Process Regressor (from `sklearn`)  
- **Search space:** Normalized to [0,1]^d for each function  
- **Acquisition functions:**  
  - EI: Expected Improvement  
  - UCB: Upper Confidence Bound (β = 2.0)  
- **Acquisition optimization:** Random sampling over the search space  

## **Submission  
Submit the following items as part of your assignment:
- Python implementation of **EI** and **UCB** using NumPy  
- Comparison plots showing **EI vs. UCB performance** for all six benchmark functions  

In [ ]:
import numpy as np
from scipy.special import erf
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, WhiteKernel, ConstantKernel

In [ ]:
# ---- Normal PDF / CDF -------------------------------------------------
def normal_pdf(z):
    """Standard normal probability density function."""
    z = np.asarray(z)
    return (1.0 / np.sqrt(2.0 * np.pi)) * np.exp(-0.5 * z**2)


def normal_cdf(z):
    """Standard normal cumulative distribution function using scipy.special.erf."""
    z = np.asarray(z)
    return 0.5 * (1.0 + erf(z / np.sqrt(2.0)))


# --- You can change here ----------------------------------------------
# ---- Acquisition functions -------------------------------------------
def expected_improvement(mu, sigma, y_best, xi=0.01):

    return np.zeros_like(mu)


def upper_confidence_bound(mu, sigma, beta=2.0):

    return np.zeros_like(mu)
# -----------------------------------------------------------------------

In [ ]:
def run_bo_numpy(
    objective_fn,
    bounds,
    n_init=5,
    n_iter=25,
    acq_type="ei",
    random_state=0,
):
    """
    Simple Bayesian Optimization loop using sklearn GP + random search over acquisition.

    Parameters
    ----------
    objective_fn : callable
        f(x) to maximize. x is 1D array of shape (d,).
    bounds : list of (low, high)
        Search box for each dimension.
    n_init : int
        Number of initial random evaluations.
    n_iter : int
        Number of BO iterations.
    acq_type : {"ei", "ucb"}
        Acquisition type.
    random_state : int
        Seed for reproducibility.

    Returns
    -------
    X_all : np.ndarray, shape (n_init + n_iter, d)
        All evaluated points.
    y_all : np.ndarray, shape (n_init + n_iter,)
        All observed objective values.
    best_history : np.ndarray, shape (n_init + n_iter,)
        Running best-so-far values.
    """
    rng = np.random.default_rng(random_state)
    d = len(bounds)

    # ---- Initial random design ----------------------------------------
    lows = np.array([b[0] for b in bounds])
    highs = np.array([b[1] for b in bounds])

    X = rng.uniform(lows, highs, size=(n_init, d))
    y = np.array([objective_fn(x) for x in X])

    best_history = [np.max(y)]

    # ---- BO iterations ------------------------------------------------
    for t in range(n_iter):
        # Fit GP to current data
        kernel = ConstantKernel(1.0, (1e-3, 1e3)) * RBF(
            length_scale=np.ones(d), length_scale_bounds=(1e-2, 1e2)
        ) + WhiteKernel(noise_level=1e-6, noise_level_bounds=(1e-10, 1e-1))

        gp = GaussianProcessRegressor(
            kernel=kernel,
            alpha=0.0,
            normalize_y=True,
            n_restarts_optimizer=3,
            random_state=random_state,
        )
        gp.fit(X, y)

        # Propose next point by random search on acquisition
        n_candidates = 1024
        X_cand = rng.uniform(lows, highs, size=(n_candidates, d))
        mu, sigma = gp.predict(X_cand, return_std=True)
        y_best = np.max(y)

        if acq_type.lower() == "ei":
            acq_values = expected_improvement(mu, sigma, y_best)
        elif acq_type.lower() == "ucb":
            acq_values = upper_confidence_bound(mu, sigma, beta=2.0)
        else:
            raise ValueError(f"Unknown acq_type: {acq_type}")

        idx_best = np.argmax(acq_values)
        x_next = X_cand[idx_best]
        y_next = objective_fn(x_next)

        # Append new data
        X = np.vstack([X, x_next[None, :]])
        y = np.concatenate([y, np.array([y_next])])

        best_history.append(np.max(y))

    return X, y, np.array(best_history)

In [ ]:
# 1. Forrester: x ∈ [0,1], usually a minimization problem
def forrester(x):
    """Forrester function, x in [0,1]."""
    x = np.asarray(x)
    return ((6.0 * x - 2.0) ** 2) * np.sin(12.0 * x - 4.0)


def forrester_max(x_vec):
    """Wrapper for BO: maximize -forrester(x). x_vec shape (1,)."""
    x = x_vec[0]
    return -forrester(x)


bounds_forrester = [(0.0, 1.0)]  # 1D

def sinus_mixture(x):
    """
    Sinusoidal mixture:
        f(x) = sin(3x) + 0.3 * sin(7x)
    """
    x = np.asarray(x)
    return np.sin(3.0 * x) + 0.3 * np.sin(7.0 * x)


def sinus_max(x_vec):
    """Wrapper: x_vec shape (1,), directly maximize sinus_mixture."""
    x = x_vec[0]
    return sinus_mixture(x)


bounds_sinus = [(0.0, 1.0)]  # you could also use (0, 2*np.pi) if you want

def branin(x):
    """Branin–Hoo function, x shape (..., 2)."""
    x = np.asarray(x)
    x1 = x[..., 0]
    x2 = x[..., 1]

    a = 1.0
    b = 5.1 / (4.0 * np.pi ** 2)
    c = 5.0 / np.pi
    r = 6.0
    s = 10.0
    t = 1.0 / (8.0 * np.pi)

    term1 = a * (x2 - b * x1 ** 2 + c * x1 - r) ** 2
    term2 = s * (1.0 - t) * np.cos(x1)
    return term1 + term2 + s


def scale_unit_to_branin(x_unit):
    """Map x in [0,1]^2 to Branin's domain."""
    x_unit = np.asarray(x_unit)
    bounds = np.array([[-5.0, 10.0],
                       [ 0.0, 15.0]])  # shape (2,2)
    low = bounds[:, 0]
    high = bounds[:, 1]
    return low + x_unit * (high - low)


def branin_max(x_unit):
    """
    Wrapper for BO: x_unit in [0,1]^2, maximize -Branin.
    x_unit: shape (2,)
    """
    x_orig = scale_unit_to_branin(x_unit)
    return -branin(x_orig)


bounds_branin = [(0.0, 1.0), (0.0, 1.0)]  # 2D unit box

def goldstein_price(x):
    """Goldstein–Price function, x shape (...,2)."""
    x = np.asarray(x)
    x1 = x[..., 0]
    x2 = x[..., 1]

    term1 = (
        1.0
        + (x1 + x2 + 1.0) ** 2
        * (
            19.0
            - 14.0 * x1
            + 3.0 * x1 ** 2
            - 14.0 * x2
            + 6.0 * x1 * x2
            + 3.0 * x2 ** 2
        )
    )
    term2 = (
        30.0
        + (2.0 * x1 - 3.0 * x2) ** 2
        * (
            18.0
            - 32.0 * x1
            + 12.0 * x1 ** 2
            + 48.0 * x2
            - 36.0 * x1 * x2
            + 27.0 * x2 ** 2
        )
    )
    return term1 * term2


def scale_unit_to_goldstein(x_unit):
    """Map x in [0,1]^2 to [-2,2]^2."""
    x_unit = np.asarray(x_unit)
    low = np.array([-2.0, -2.0])
    high = np.array([ 2.0,  2.0])
    return low + x_unit * (high - low)


def goldstein_max(x_unit):
    """
    Wrapper for BO: x_unit in [0,1]^2, maximize -Goldstein.
    """
    x_orig = scale_unit_to_goldstein(x_unit)
    return -goldstein_price(x_orig)


bounds_goldstein = [(0.0, 1.0), (0.0, 1.0)]

def hartmann3(x):
    """
    Hartmann 3D function, x in [0,1]^3.
    Typically used as minimization.
    """
    x = np.asarray(x)
    alpha = np.array([1.0, 1.2, 3.0, 3.2])

    A = np.array(
        [
            [3.0, 10.0, 30.0],
            [0.1, 10.0, 35.0],
            [3.0, 10.0, 30.0],
            [0.1, 10.0, 35.0],
        ]
    )

    P = 1e-4 * np.array(
        [
            [3689, 1170, 2673],
            [4699, 4387, 7470],
            [1091, 8732, 5547],
            [381, 5743, 8828],
        ]
    )

    x_expanded = x[..., None, :]            # (..., 1, 3)
    diff = x_expanded - P                   # (..., 4, 3)
    inner = np.sum(A * diff ** 2, axis=-1)  # (..., 4)
    outer = np.sum(alpha * np.exp(-inner), axis=-1)

    return -outer  # more negative is better (for minimization)


def hartmann3_max(x_vec):
    """Wrapper for BO: x_vec shape (3,), maximize -hartmann3(x)."""
    return -hartmann3(x_vec)  # = +outer


bounds_hart3 = [(0.0, 1.0)] * 3

def hartmann6(x):
    """
    Hartmann 6D function, x in [0,1]^6.
    """
    x = np.asarray(x)
    alpha = np.array([1.0, 1.2, 3.0, 3.2])

    A = np.array(
        [
            [10.0, 3.0, 17.0, 3.5, 1.7, 8.0],
            [0.05, 10.0, 17.0, 0.1, 8.0, 14.0],
            [3.0, 3.5, 1.7, 10.0, 17.0, 8.0],
            [17.0, 8.0, 0.05, 10.0, 0.1, 14.0],
        ]
    )

    P = 1e-4 * np.array(
        [
            [1312, 1696, 5569, 124, 8283, 5886],
            [2329, 4135, 8307, 3736, 1004, 9991],
            [2348, 1451, 3522, 2883, 3047, 6650],
            [4047, 8828, 8732, 5743, 1091, 381],
        ]
    )

    x_expanded = x[..., None, :]            # (..., 1, 6)
    diff = x_expanded - P                   # (..., 4, 6)
    inner = np.sum(A * diff ** 2, axis=-1)
    outer = np.sum(alpha * np.exp(-inner), axis=-1)

    return -outer


def hartmann6_max(x_vec):
    """Wrapper for BO: x_vec shape (6,), maximize -hartmann6(x)."""
    return -hartmann6(x_vec)


bounds_hart6 = [(0.0, 1.0)] * 6



In [ ]:
import matplotlib.pyplot as plt

# Compare all problems visually
problems = [
    ("Forrester 1D",   forrester_max,   bounds_forrester),
    ("Sinus 1D",       sinus_max,       bounds_sinus),
    ("Branin 2D",      branin_max,      bounds_branin),
    ("Goldstein 2D",   goldstein_max,   bounds_goldstein),
    ("Hartmann3 3D",   hartmann3_max,   bounds_hart3),
    ("Hartmann6 6D",   hartmann6_max,   bounds_hart6),
]

plt.figure(figsize=(12, 10))

for i, (name, obj, bounds) in enumerate(problems, 1):
    # --- Run BO with both acquisitions ---
    X_ei, y_ei, best_ei = run_bo_numpy(
        objective_fn=obj,
        bounds=bounds,
        n_init=5,
        n_iter=25,
        acq_type="ei",
        random_state=0,
    )

    X_ucb, y_ucb, best_ucb = run_bo_numpy(
        objective_fn=obj,
        bounds=bounds,
        n_init=5,
        n_iter=25,
        acq_type="ucb",
        random_state=0,
    )

    # --- Plot best-so-far curve ---
    plt.subplot(3, 2, i)
    plt.plot(best_ei, label="EI", linewidth=2)
    plt.plot(best_ucb, label="UCB", linewidth=2, linestyle="--")
    plt.title(name)
    plt.xlabel("Iteration")
    plt.ylabel("Best Value so far")
    plt.legend()
    plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()